## 1. Installazione Dipendenze

Esegui questa cella per installare tutti i pacchetti necessari:

In [1]:
# Installa le dipendenze necessarie
!pip install anthropic Pillow PyPDF2

## 2. Importa il Sistema

Assicurati che il file `archival_llm_system_fixed.py` sia nella stessa directory del notebook, oppure specifica il percorso completo.

In [2]:
# Importa il sistema (assicurati che archival_llm_system_fixed.py sia nella stessa directory)
from archival_llm_system_fixed import Orchestrator

# Se il file è in un'altra directory, aggiungi il path:
# import sys
# sys.path.append('/path/to/directory')
# from archival_llm_system_fixed import Orchestrator

## 3. Configurazione

Configura i parametri del sistema:

In [ ]:
# Configurazione API Key
API_KEY = ""  # INSERISCI QUI LA TUA API KEY

# Path ai file e cartelle
METADATI_ESSENZIALI = r"05_metadati_essenziali.json"
METADATI_COMPLETI = r"05_metadati_completi.json"
CARTELLA_IMMAGINI = r"05.1303"
LINEE_GUIDA_METS = r"ICDP_Profilo_METS_ECO-MiC_v.1.2.pdf"  # Opzionale

# Configurazione preprocessing
PREPROCESS_IMAGES = True       # True = converti in B&W e aumenta contrasto
CONTRAST_FACTOR = 2.0          # Fattore di aumento contrasto (1.0-3.0)
SAVE_PREVIEW = True            # True = salva preview delle immagini preprocessate
PREVIEW_FOLDER = "./preview"   # Cartella per le preview

# Configurazione output
GENERA_METS = True             # True = genera anche XML-METS
OUTPUT_METS_PATH = "output_mets.xml"
OUTPUT_JSON_PATH = "output_trascrizione.json"

## 4. Inizializza l'Orchestratore

In [4]:
# Crea l'orchestratore con le configurazioni
orchestrator = Orchestrator(
    llm_provider="anthropic",
    api_key=API_KEY,
    preprocess_images=PREPROCESS_IMAGES,
    contrast_factor=CONTRAST_FACTOR,
    save_preview=SAVE_PREVIEW,
    preview_folder=PREVIEW_FOLDER,
    use_prompt_caching=True,  # ✅ Mantieni sempre True per ridurre i costi
    linee_guida_mets_path=LINEE_GUIDA_METS if GENERA_METS else None
)

print("✅ Orchestratore inizializzato con successo!")


💰 PROMPT CACHING ATTIVO
[agente_mets_formatter] Linee guida PDF caricate: 269255 caratteri (estratti)
✅ Orchestratore inizializzato con successo!


## 5. Esegui il Processo Completo

Questa cella esegue tutte le fasi: analisi, trascrizione, regesto e (opzionalmente) generazione METS.

In [5]:
try:
    # Esegui il processo completo
    output = orchestrator.process_manuscript(
        metadati_file=METADATI_ESSENZIALI,
        cartella_immagini=CARTELLA_IMMAGINI,
        metadati_completi_file=METADATI_COMPLETI,
        genera_mets=GENERA_METS,
        output_mets_path=OUTPUT_METS_PATH
    )
    
    print("\n" + "="*60)
    print("✅ PROCESSO COMPLETATO CON SUCCESSO!")
    print("="*60)
    
except Exception as e:
    print(f"\n❌ ERRORE: {e}")
    import traceback
    traceback.print_exc()

INIZIO ORCHESTRAZIONE

📁 Caricate 4 immagini:
  1. MC0069_CNMD0000424589_00001.jpg
  2. MC0069_CNMD0000424589_00002.jpg
  3. MC0069_CNMD0000424589_00003.jpg
  4. MC0069_CNMD0000424589_00004.jpg

🖼️  PREPROCESSING ATTIVO:
  - Conversione in bianco e nero
  - Aumento contrasto: 2.0x
  - Preview salvate in: ./preview/

FASE 1: ANALISI (crea cache)

[agente_analisi] Inizio analisi del manoscritto...
[agente_analisi] Analizzando 4 immagini

[API CALL #1] Chiamata vision API
[LLM] Preprocessing: ATTIVO
[LLM] Contrasto: 2.0x
[LLM] 💾 Prompt Caching: ATTIVO
[LLM] Immagine 1/4: MC0069_CNMD0000424589_00001.jpg
[LLM] Immagine 2/4: MC0069_CNMD0000424589_00002.jpg
[LLM] Immagine 3/4: MC0069_CNMD0000424589_00003.jpg
[LLM] ✓ Cache abilitata per 4 immagini
[LLM] Immagine 4/4: MC0069_CNMD0000424589_00004.jpg
[LLM] ✓ Cache abilitata per system prompt
[agente_analisi] Risposta LLM ricevuta
[agente_analisi] Risultati scritti in memoria

FASE 2: TRASCRIZIONE (usa cache + validazione metadati esterni)

[agen

## 6. Visualizza il Report

In [6]:
# Stampa il report completo
orchestrator.print_report(output)


REPORT FINALE

📊 METADATI DESCRITTIVI ANALIZZATI:
  lingua: italiano moderno
  tipologia_documento: lettera privata
  natura_documento: autografo
  tipo_scrittura: corsivo moderno
  abbreviazioni: ['Tel. = Telefono', 'Cav. Uff. = Cavaliere Ufficiale', 'FR. = Francia']
  aree_del_testo: ['intestazione hotel con fotografia edificio', 'informazioni servizi hotel (colonna sinistra e destr...
  descrizione_elementi: ["Carta intestata Hotel Crammont et Des Alpes, Pre-St-Didier, Valle d'Aosta", "Fotografia in bianco ...
  particolarità_linguistiche: {'Aosta': 'sottolineato nella parola (Aosta) nel testo manoscritto', 'fisie': "possibile errore di s...
  composizione_oggetto: Immagine 1: fronte della lettera con carta intestata dell'hotel, fotografia, testo manoscritto e fir...

🔧 METADATI TECNICI:
  Numero immagini: 4
  Dettagli immagini disponibili: 4

📄 TRASCRIZIONE:
  <archivaldescription>05.1303</archivaldescription>

<!-- Pagina 1: Carta intestata dell'hotel con testo manoscritto -->

<

## 7. Salva i Risultati

In [7]:
import json

# Salva l'output completo come JSON
with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"✅ Output salvato in: {OUTPUT_JSON_PATH}")

if GENERA_METS and "mets_xml" in output:
    print(f"✅ XML-METS salvato in: {OUTPUT_METS_PATH}")

✅ Output salvato in: output_trascrizione.json
✅ XML-METS salvato in: output_mets.xml


## 8. Ispeziona i Risultati

Puoi ispezionare specificamente singoli componenti dell'output:

In [8]:
# Visualizza solo la trascrizione
print("TRASCRIZIONE:")
print("="*60)
print(output['trascrizione'])

TRASCRIZIONE:
<archivaldescription>05.1303</archivaldescription>

<!-- Pagina 1: Carta intestata dell'hotel con testo manoscritto -->

<transcription>

PRE-ST-DIDIER - VALLE D'AOSTA

FERROVIA
AOSTA - PRE ST-DIDIER:
COLLEGATA ALLA RETE
NAZIONALE
ED INTERNAZIONALE

AUTOPULLMAN
MILANO
TORINO
GENOVA

AUTOCORRIERE
COURMAYEUR
LA THUILE

PICCOLO S. BERNARDO
BOURG ST. MAURICE
(FR.)

ALTITUDINE
METRI 1010

GUIDE E
PORTATORI DEL CAI

BELLE PASSEGGIATE
TRA PINETE
E
DENSE PRATERIE

SORGENTE
TERMALE
FERRUGINOSA
CARBONICA
ARSENICALE
RADIOATTIVA

Hôtel Crammont et Des Alpes
Bar Chez Catherine

Telefono 432

AL BIVIO DELLA STRADA INTERNAZIONALE AOSTA - PICCOLO S. BERNARDO - BOURG ST. MAURICE E STRADA AOSTA - COURMAYEUR - APERTO TUTTO L'ANNO - RITROVO SPORTIVO INVERNALE - POSTO TELEFONICO PUBBLICO - FRESCO SOGGIORNO ESTIVO - TRATTAMENTO FAMILIARE - GRANDE VERANDA BELVEDERE CON TOTALE VISIONE DELLE CATENE DEL MONTE BIANCO - TENNIS - GIARDINO - BAR - SALE E SALOTTI SOGGIORNO - ACQUA CORRENTE CALDA E FRED

In [9]:
# Visualizza solo il regesto
if 'regesto' in output:
    print("REGESTO:")
    print("="*60)
    print(output['regesto'])
    print("\nFonti utilizzate:")
    print(json.dumps(output.get('regesto_fonti_utilizzate', {}), indent=2, ensure_ascii=False))

REGESTO:
Sibilla Aleramo scrive al soldato Elio Fiore il 23 luglio 1959 da Pré-St-Didier (Valle d'Aosta), dove soggiorna presso l'Hotel Crammont. Si informa su come il destinatario stia trascorrendo gli ultimi giorni con la divisa militare e gli chiede di scriverle. Esprime il desiderio di rimanere in montagna fino a fine agosto per recuperare le forze, lamentando una grande debolezza fisica. Chiede notizie della madre di Elio e si augura che egli affronti serenamente il nuovo periodo di vita da soldato.

Fonti utilizzate:
{
  "mittente": "metadati_esterni",
  "destinatario": "trascrizione_tag_xml",
  "data": "metadati_esterni",
  "luogo": "metadati_esterni",
  "contenuto": "trascrizione_testo",
  "tipologia": "analisi_paleografica"
}


In [10]:
# Visualizza i metadati analizzati
print("METADATI ANALIZZATI DALL'LLM:")
print("="*60)
print(json.dumps(output['metadati_descrittivi_LLM'], indent=2, ensure_ascii=False))

METADATI ANALIZZATI DALL'LLM:
{
  "lingua": "italiano moderno",
  "tipologia_documento": "lettera privata",
  "natura_documento": "autografo",
  "tipo_scrittura": "corsivo moderno",
  "abbreviazioni": [
    "Tel. = Telefono",
    "Cav. Uff. = Cavaliere Ufficiale",
    "FR. = Francia"
  ],
  "aree_del_testo": [
    "intestazione hotel con fotografia edificio",
    "informazioni servizi hotel (colonna sinistra e destra)",
    "corpo della lettera manoscritta",
    "firma finale",
    "busta con indirizzo destinatario",
    "timbri postali sulla busta",
    "numero archivistico in alto a destra"
  ],
  "descrizione_elementi": [
    "Carta intestata Hotel Crammont et Des Alpes, Pre-St-Didier, Valle d'Aosta",
    "Fotografia in bianco e nero dell'hotel con montagne sullo sfondo",
    "Elenco servizi hotel in colonne laterali (ferrovia, autopullman, guide, passeggiate, sorgenti termali, etc.)",
    "Numero archivistico '05.1303' scritto a mano in alto a destra",
    "Busta con due timbri pos

In [11]:
# Visualizza eventuali correzioni applicate
if 'trascrizione_correzioni_applicate' in output:
    print("CORREZIONI APPLICATE:")
    print("="*60)
    for correzione in output['trascrizione_correzioni_applicate']:
        print(f"• {correzione}")

## 10. Debug: Visualizza Storia Modifiche

Per capire cosa ha fatto ogni agente:

In [13]:
# Visualizza la storia completa delle modifiche
storia = orchestrator.memory.get_storia()

print(f"STORIA MODIFICHE ({len(storia)} eventi)")
print("="*60)

for i, evento in enumerate(storia, 1):
    print(f"\n{i}. [{evento['timestamp']}]")
    print(f"   Agente: {evento['agente']}")
    print(f"   Azione: {evento['azione']}")
    if 'dettagli' in evento and evento['dettagli']:
        print(f"   Dettagli: {str(evento['dettagli'])[:100]}...")

STORIA MODIFICHE (13 eventi)

1. [2026-03-02T15:41:33.521005]
   Agente: sistema
   Azione: caricamento_metadati_esterni
   Dettagli: {'segnatura': 'CNMD\\0000424589', 'data_creazione': '1959-07-23', 'materia': 'Cartaceo', 'numero_car...

2. [2026-03-02T15:41:33.525186]
   Agente: sistema
   Azione: registrazione_immagini
   Dettagli: {'numero_immagini': 4}...

3. [2026-03-02T15:41:56.083206]
   Agente: agente_analisi
   Azione: aggiornamento_lingua
   Dettagli: {'valore_nuovo': 'italiano moderno', 'valore_precedente': None, 'confidence': 0.95, 'note': 'Testo i...

4. [2026-03-02T15:41:56.083206]
   Agente: agente_analisi
   Azione: aggiornamento_tipologia_documento
   Dettagli: {'valore_nuovo': 'lettera privata', 'valore_precedente': None, 'confidence': 0.98, 'note': "Lettera ...

5. [2026-03-02T15:41:56.083206]
   Agente: agente_analisi
   Azione: aggiornamento_natura_documento
   Dettagli: {'valore_nuovo': 'autografo', 'valore_precedente': None, 'confidence': 0.95, 'note': 'Scrittur

## 11. (Opzionale) Elaborazione Batch di Più Documenti

Se hai più documenti da processare:

In [14]:
import os

# Lista di documenti da processare
documenti = [
    {
        "nome": "documento_001",
        "metadati": r"C:\path\to\doc1\metadati.json",
        "immagini": r"C:\path\to\doc1\immagini",
        "metadati_completi": r"C:\path\to\doc1\metadati_completi.json"
    },
    {
        "nome": "documento_002",
        "metadati": r"C:\path\to\doc2\metadati.json",
        "immagini": r"C:\path\to\doc2\immagini",
        "metadati_completi": r"C:\path\to\doc2\metadati_completi.json"
    },
    # ... altri documenti
]

# Crea cartella output se non esiste
output_dir = "batch_output"
os.makedirs(output_dir, exist_ok=True)

# Processa tutti i documenti
risultati = []

for i, doc in enumerate(documenti, 1):
    print(f"\n{'='*60}")
    print(f"PROCESSANDO {i}/{len(documenti)}: {doc['nome']}")
    print(f"{'='*60}\n")
    
    try:
        # ⚠️ IMPORTANTE: Crea un nuovo orchestratore per ogni documento
        # per evitare conflitti nella memoria condivisa
        orch = Orchestrator(
            llm_provider="anthropic",
            api_key=API_KEY,
            preprocess_images=PREPROCESS_IMAGES,
            contrast_factor=CONTRAST_FACTOR,
            save_preview=SAVE_PREVIEW,
            preview_folder=f"{PREVIEW_FOLDER}/{doc['nome']}",
            use_prompt_caching=True,
            linee_guida_mets_path=LINEE_GUIDA_METS if GENERA_METS else None
        )
        
        output = orch.process_manuscript(
            metadati_file=doc['metadati'],
            cartella_immagini=doc['immagini'],
            metadati_completi_file=doc.get('metadati_completi'),
            genera_mets=GENERA_METS,
            output_mets_path=f"{output_dir}/{doc['nome']}_mets.xml"
        )
        
        # Salva JSON
        with open(f"{output_dir}/{doc['nome']}_output.json", "w", encoding="utf-8") as f:
            json.dump(output, f, indent=2, ensure_ascii=False)
        
        risultati.append({
            "nome": doc['nome'],
            "stato": "✅ Completato",
            "output_path": f"{output_dir}/{doc['nome']}_output.json"
        })
        
        print(f"\n✅ {doc['nome']} completato con successo!")
        
    except Exception as e:
        risultati.append({
            "nome": doc['nome'],
            "stato": f"❌ Errore: {str(e)}",
            "output_path": None
        })
        print(f"\n❌ Errore su {doc['nome']}: {e}")

# Riepilogo finale
print("\n" + "="*60)
print("RIEPILOGO BATCH")
print("="*60)
for r in risultati:
    print(f"{r['stato']} - {r['nome']}")
    if r['output_path']:
        print(f"  Output: {r['output_path']}")


PROCESSANDO 1/2: documento_001


💰 PROMPT CACHING ATTIVO
[agente_mets_formatter] Linee guida PDF caricate: 269255 caratteri (estratti)
INIZIO ORCHESTRAZIONE

❌ Errore su documento_001: [Errno 2] No such file or directory: 'C:\\path\\to\\doc1\\metadati.json'

PROCESSANDO 2/2: documento_002


💰 PROMPT CACHING ATTIVO
[agente_mets_formatter] Linee guida PDF caricate: 269255 caratteri (estratti)
INIZIO ORCHESTRAZIONE

❌ Errore su documento_002: [Errno 2] No such file or directory: 'C:\\path\\to\\doc2\\metadati.json'

RIEPILOGO BATCH
❌ Errore: [Errno 2] No such file or directory: 'C:\\path\\to\\doc1\\metadati.json' - documento_001
❌ Errore: [Errno 2] No such file or directory: 'C:\\path\\to\\doc2\\metadati.json' - documento_002


## 📋 Note Importanti

### Costi API
- Il sistema usa **Prompt Caching** per ridurre i costi
- La prima chiamata costa di più, le successive sono molto più economiche
- Stimare circa:
  - Analisi: ~$0.10-0.30 per documento (4 immagini)
  - Trascrizione: ~$0.05-0.15 (usa cache)
  - Regesto: ~$0.01-0.02 (solo text)
  - METS: ~$0.02-0.05 (solo text)

### Performance
- Preprocessing immagini: +30-60 secondi per documento
- Chiamate API: 5-15 secondi ciascuna
- Totale per documento: ~2-4 minuti

### Troubleshooting
- Se ottieni errori di parsing JSON → controlla `debug_llm_response.txt`
- Se le trascrizioni sono imprecise → aumenta `CONTRAST_FACTOR` a 2.5-3.0
- Se le immagini sono troppo grandi → il sistema le ridimensiona automaticamente a <5MB

### Requisiti
- Python 3.8+
- Anthropic API key valida
- 500MB+ RAM per immagini grandi
- Connessione internet stabile